> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Synthetic — end-to-end complex mechanistic integration test

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import vigor
print("VIGOR", vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__
SEED=42
cfg=vigor.simulation_scenario("complex")
cfg.update(dict(heritability=.55, compute_grms=True))
sim=vigor.simulate_genomic_dataset(n_samples=320,n_variants=1000,n_traits=8,seed=SEED,genotype_kwargs={"n_chromosomes":5,"generations":100},trait_kwargs=cfg)
X=sim.genotypes.to_pandas(); Y=sim.phenotypes.to_pandas()
Ka=sim.grms.sel(kernel="additive").to_pandas().to_numpy(float); Kd=sim.grms.sel(kernel="dominance").to_pandas().to_numpy(float)
obs=[c for c in sim.environment.to_pandas() if str(c).startswith("observed")][0]
e=sim.environment.to_pandas()[obs].to_numpy(float); Kg=Ka*np.outer(e,e); Kg/=np.mean(np.diag(Kg))
kernels={"additive":Ka,"dominance":Kd,"gxe":Kg}
rng=np.random.default_rng(20260918); perm=rng.permutation(len(Y)); fit_idx=perm[:224]; unseen_idx=perm[224:272]; partial_idx=perm[272:]
fitmask=np.zeros(len(Y),bool); fitmask[fit_idx]=True
train=Y.copy(); train.iloc[~fitmask,:]=np.nan
pred_traits=train.copy(); held=np.zeros(Y.shape,bool)
for i in partial_idx:
    reveal=rng.choice(Y.shape[1],Y.shape[1]//2,replace=False); pred_traits.iloc[i,reveal]=Y.iloc[i,reveal]; held[i,:]=True; held[i,reveal]=False
fit=vigor.genetic_varcov_PSD(kernels,train,rank_G={k:Y.shape[1] for k in kernels},rank_E=Y.shape[1],solver="lbfgs",variance_closure="auto",variance_closure_strength=1.0,analytic_se="final",analytic_se_probes=16,analytic_trace_probes=8,n_boot=0,random_state=SEED,progress=False,blas_threads=2)

pred=vigor.mvBLUP(pred_traits,kernels,fit,E=None,missing="exact",accuracy="exact",kernel_rank="full",blas_threads=2)["total"].to_pandas()
truthG=sim.components.sel(component="G_total").to_pandas().to_numpy(); unseen=np.repeat(np.isin(np.arange(len(Y)),unseen_idx)[:,None],Y.shape[1],1)
r2=np.corrcoef(truthG[unseen],np.asarray(pred)[unseen])[0,1]**2
print({"scenario":"complex","N":len(Y),"M":X.shape[1],"T":Y.shape[1],"unseen_genetic_R2":float(r2),"closure_MAE":float(np.mean(np.abs(np.asarray(fit["variance_closure_ratio"].to_pandas())-1)))})
